In [1]:
import json

with open("/content/drive/MyDrive/Colab Notebooks/test.jsonl", "r") as f:
    for line in f:
        obj = json.loads(line)
        #print(obj)

In [ ]:
!mkdir onnx-out
!mkdir awq-out
!mkdir temp

In [ ]:
#!git clone https://huggingface.co/onnx-community/Qwen3-0.6B-ONNX
!git clone https://huggingface.co/FreedomIntelligence/Apollo2-1.5B

Cloning into 'Apollo2-1.5B'...
remote: Enumerating objects: 52, done.
remote: Total 52 (delta 0), reused 0 (delta 0), pack-reused 52 (from 1)
Receiving objects: 100% (52/52), 3.62 MiB | 12.63 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [ ]:
#!git clone https://github.com/casper-hansen/AutoAWQ
#!cd AutoAWQ
#!pip install -e .
!pip install autoawq
!pip install onnx_ir==0.2.0
!pip install onnxruntime-genai==0.12.0
!pip install transformers==4.51.3 -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=59a2d895a42bb1c6d321c3b54af388cf5918c5f7fc08f9c339408528810bfdab
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hu

In [ ]:
from datasets import load_dataset
data = load_dataset(
    "wikitext",'wikitext-103-raw-v1',streaming=True)["train"]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
import pandas as pd

patients = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/patients.csv")
conditions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/conditions.csv")
medications = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/medications.csv")
encounters = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/apollo2v3/csv/encounters.csv")

In [ ]:
cond_map = conditions.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()
med_map = medications.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()
enc_map = encounters.groupby("PATIENT")["DESCRIPTION"].apply(list).to_dict()

In [ ]:
import random

def generate_tasks(pid, patient):
    conds = cond_map.get(pid, [])
    meds = med_map.get(pid, [])
    encs = enc_map.get(pid, [])

    age = 2024 - int(patient["BIRTHDATE"][:4])
    gender = patient["GENDER"]

    base_text = f"{age}-year-old {gender}. Conditions: {', '.join(conds[:5])}."

    tasks = []

    # Task 1: Summary
    tasks.append({
        "instruction": "Summarize the patient's medical condition",
        "input": base_text,
        "output": f"The patient has {', '.join(conds[:3])}."
    })

    # Task 2: Diagnosis
    tasks.append({
        "instruction": "What conditions does this patient have?",
        "input": base_text,
        "output": ", ".join(conds[:5])
    })

    # Task 3: Medication reasoning
    if meds:
        tasks.append({
            "instruction": "What medications is the patient taking?",
            "input": base_text,
            "output": ", ".join(meds[:5])
        })

    # Task 4: Encounter reasoning
    if encs:
        tasks.append({
            "instruction": "What type of medical visits has the patient had?",
            "input": base_text,
            "output": ", ".join(encs[:3])
        })

    # Task 5: Clinical note
    tasks.append({
        "instruction": "Write a brief clinical note",
        "input": base_text,
        "output": f"{age}-year-old {gender} with history of {', '.join(conds[:3])}."
    })

    return tasks

In [ ]:
dataset = []

for _, patient in patients.iterrows():
    pid = patient["Id"]

    tasks = generate_tasks(pid, patient)

    # repeat with variation
    for _ in range(10):  # key multiplier
        for t in tasks:
            # slight variation
            t_copy = t.copy()
            if random.random() > 0.5:
                t_copy["instruction"] += " Please be concise."

            dataset.append(t_copy)

# limit to ~200K
dataset = dataset[:200000]

print(len(dataset))

180000


In [ ]:
from datasets import load_dataset
#hf_dataset = Dataset.from_list(dataset)
#hf_dataset.save_to_disk("synthea_200k")

dataset = load_dataset(
    "medmcqa",
    split="train"
)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

KeyError: 'train'

In [ ]:
print(dataset[0])
print(len(dataset))
prompt = """Question: {question}
1. {A}
2. {B}
3. {C}
4. {D}
Answer: {explaination}, hence correct option is {correct_idx}.
"""
calibration_texts=[]
for sample in dataset.select(range(182822)):
  calibration_texts.append(prompt.format(question = sample['question'],A = sample['opa'],B= sample['opb'],C = sample['opc'],
                                         D = sample['opd'], explaination = sample['exp'], correct_idx=sample['cop']))

print(calibration_texts[0])

{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}
182822
Question: Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma
1. Hyperplasia
2. Hyperophy
3. Atrophy
4. Dyplasia
Answer: Chronic urethral obstruction because of urinary calculi, pro

In [ ]:
print(dataset[0])
print(type(dataset[0]))

{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}
<class 'dict'>


In [ ]:
from datasets import load_from_disk
ds = load_from_disk("synthea_200k")
print(ds)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 180000
})


In [ ]:
print(ds[100])
prompt = '''{instruction}. The basic details of patient: {input} Assistant: {output}
'''
calibration_texts=[]
for sample in ds:
  calibration_texts.append(prompt.format(instruction = sample['instruction'],input = sample['input'],output = sample['output']))

{'instruction': "Summarize the patient's medical condition Please be concise.", 'input': '41-year-old M. Conditions: Adolescent idiopathic scoliosis (disorder), Received higher education (finding), Prediabetes (finding), Anemia (disorder), Chronic intractable migraine without aura (disorder).', 'output': 'The patient has Adolescent idiopathic scoliosis (disorder), Received higher education (finding), Prediabetes (finding).'}


In [ ]:
# https://raw.githubusercontent.com/microsoft/onnxruntime-genai/main/examples/python/awq-quantized-model.py
import json
import os

from awq import AutoAWQForCausalLM
from onnxruntime_genai.models.builder import create_model
from transformers import AutoTokenizer


model_path = '/content/Apollo2'
quant_path = '/content/awq-out'
output_path = '/content/onnx-out'
execution_provider = "cpu"

quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

# Load model
model = AutoAWQForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, use_cache=False)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Quantize model
model.quantize(tokenizer, quant_config=quant_config, calib_data =calibration_texts)

# Save quantized model
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

print(f'Model is quantized and saved at "{quant_path}"')



/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

Model is quantized and saved at "/content/awq-out"


In [ ]:
from onnxruntime_genai.models.builder import create_model

# Create ONNX model
model_name = None
input_folder = '/content/awq-out'
output_folder = '/content/onnx-out'
precision = "int4"
execution_provider = "cpu"
cache_dir = '/content/temp'

create_model(model_name, input_folder, output_folder, precision, execution_provider, cache_dir)


GroupQueryAttention (GQA) is used in this model.
Unpacking and repacking layer 0
Unpacking and repacking layer 1
Unpacking and repacking layer 2
Unpacking and repacking layer 3
Unpacking and repacking layer 4
Unpacking and repacking layer 5
Unpacking and repacking layer 6
Unpacking and repacking layer 7
Unpacking and repacking layer 8
Unpacking and repacking layer 9
Unpacking and repacking layer 10
Unpacking and repacking layer 11
Unpacking and repacking layer 12
Unpacking and repacking layer 13
Unpacking and repacking layer 14
Unpacking and repacking layer 15
Unpacking and repacking layer 16
Unpacking and repacking layer 17
Unpacking and repacking layer 18
Unpacking and repacking layer 19
Unpacking and repacking layer 20
Unpacking and repacking layer 21
Unpacking and repacking layer 22
Unpacking and repacking layer 23
Unpacking and repacking layer 24
Unpacking and repacking layer 25
Unpacking and repacking layer 26
Unpacking and repacking layer 27
Reading embedding layer
Reading decod

Saving model.embed_tokens.weight (f32, [151936,1536]): 100%|██████████| 510/510 [01:03<00:00,  8.07it/s]


Saving GenAI config in /content/onnx-out
Saving processing files in /content/onnx-out for GenAI


In [ ]:
!cp -r "onnx-out" "/content/drive/MyDrive/Colab Notebooks/apollo2v3"
#!pip freeze > requirements.txt

In [ ]:
!cp -r "output/csv" "/content/drive/MyDrive/Colab Notebooks/apollo2v3"


In [ ]:
import onnxruntime_genai as og
import json

# Load model
print("Loading model...")
config = og.Config('/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/')
#config = og.Config('/content/onnx-out/')
config.clear_providers()
model = og.Model(config)
print("Model loaded")
tokenizer = og.Tokenizer(model)

# Override any default search options in `genai_config.json`
search_options = {
        "diversity_penalty": 0.0,
        "do_sample": True,
        "early_stopping": True,
        "length_penalty": 1.0,
        "max_length": 200,
        "min_length": 1,
        "no_repeat_ngram_size": 0,
        "num_beams": 1,
        "num_return_sequences": 1,
        "past_present_share_buffer": True,
        "repetition_penalty": 1.1,
        "temperature": 0.7,
        "top_k": 40,
        "top_p": 0.9
}

params = og.GeneratorParams(model)

params.set_search_options(
    max_length=128,
    temperature=0.0
)

generator = og.Generator(model, params)

prompt = "What is diabetes?"

input_tokens = tokenizer.encode(prompt)

in_len = len(input_tokens)

generator.append_tokens(input_tokens)

while not generator.is_done():
    generator.generate_next_token()

output_tokens = generator.get_sequence(0)

text = tokenizer.decode(output_tokens[in_len:])

print(text)


Loading model...
Model loaded
 Diabetes is a chronic disease that affects the body's ability to regulate blood sugar levels. It is caused by the body producing too much insulin, a hormone that helps cells absorb glucose from the bloodstream. When there is too much insulin, the body cannot properly use the glucose, leading to high blood sugar levels. This condition is known as hyperglycemia. If the body cannot produce enough insulin, it is known as hypoglycemia. Diabetes can be classified into two types: Type 1 and Type 2. Type 1 diabetes is caused by the body's immune system attacking the insulin-producing cells in the


In [ ]:
import numpy as np
logits = generator.get_logits()
logits = np.array(logits)
print(np.sum(logits,axis=2))

In [ ]:
import time
import json
import numpy as np

import onnxruntime_genai as og

from datasets import load_dataset
from tqdm import tqdm

# =========================
# CONFIG
# =========================

MODEL_PATH = './onnx-out/'

EVAL_SAMPLES = 500
MAX_LENGTH = 512

OUTPUT_FILE = "onnx_benchmark_results.json"

# =========================
# LOAD MODEL
# =========================

print("Loading ONNX model...")

model = og.Model(MODEL_PATH)

tokenizer = og.Tokenizer(model)

# =========================
# PROMPT FORMAT
# =========================

def format_medmcqa(sample):

    options = [
        sample["opa"],
        sample["opb"],
        sample["opc"],
        sample["opd"]
    ]

    prompt = f"""
Question:
{sample['question']}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Answer:
"""

    return prompt


def extract_letter(text):

    for letter in ["A", "B", "C", "D"]:
        if letter in text:
            return letter

    return "A"

# =========================
# GENERATION FUNCTION
# =========================

def generate_answer(prompt):

    tokens = tokenizer.encode(prompt)
    input_length = len(tokens)

    # Create fresh params per request
    params = og.GeneratorParams(model)

    params.set_search_options(
        max_length=input_length + MAX_LENGTH,
        temperature=0.0,
        do_sample=False
    )

    generator = og.Generator(model, params)


    start = time.time()
    generator.append_tokens(tokens)

    while not generator.is_done():
            generator.generate_next_token()

    end = time.time()

    output_tokens = generator.get_sequence(0)

    text = tokenizer.decode(output_tokens)

    answer = text[len(prompt):].strip()

    latency = end - start

    #token_count = len(output_tokens)

    return answer, latency, input_length


# =========================
# MEDMCQA BENCHMARK
# =========================

print("\nRunning MedMCQA...")
results=np.zeros((500,3))
dataset = load_dataset(
    "medmcqa",
    split="validation"
)

correct = 0
tokens_total = 0

for sample in tqdm(dataset.select(range(EVAL_SAMPLES))):

    prompt = format_medmcqa(sample)

    answer_text, latency, tokens = generate_answer(prompt)

    pred_letter = extract_letter(answer_text)

    correct_idx = sample["cop"]

    correct_letter = ["A","B","C","D"][correct_idx]

    if pred_letter == correct_letter:
        correct += 1


medmcqa_accuracy = correct / EVAL_SAMPLES

print("medmcqa_accuracy = ", medmcqa_accuracy )

# =========================
# PUBMEDQA BENCHMARK
# =========================

print("\nRunning PubMedQA...")

pubmedqa = load_dataset(
    "pubmed_qa",
    "pqa_labeled",
    split="train"
)
results=np.zeros((500,3))
correct = 0
total = 0
for sample in pubmedqa.select(range(EVAL_SAMPLES)):

    context = " ".join(
        sample["context"]["contexts"]
    )

    question = sample["question"]

    prompt = f"""
Context:
{context}

Question:
{question}

Answer yes, no, or maybe.

Answer:
"""

    answer_text, latency, tokens = generate_answer(prompt)

    prediction = answer_text.lower()

    gold = sample["final_decision"].lower()

    if gold in prediction:
        correct += 1
        results[total,0]=1

    results[total,1]=latency
    results[total,2]=tokens
    total += 1
    print(correct,"/",total,"| tokens=",tokens)

np.save('pubmedqa_results.npy', results)
pubmedqa_accuracy = correct / EVAL_SAMPLES
print("pubmedqa_accuracy = ", pubmedqa_accuracy )
# =========================
# PERFORMANCE METRICS
# =========================

avg_latency = np.mean(latencies)

tokens_per_sec = tokens_total / np.sum(latencies)

# =========================
# SAVE RESULTS
# =========================

results = {

    "model_type": "onnxruntime-genai",

    "model_path": MODEL_PATH,

    "medmcqa_accuracy": medmcqa_accuracy,

    "pubmedqa_accuracy": pubmedqa_accuracy,

    "avg_latency_sec": float(avg_latency),

    "tokens_per_sec": float(tokens_per_sec)
}

with open(OUTPUT_FILE, "w") as f:

    json.dump(results, f, indent=4)

print("\nFinal Results:")

print(json.dumps(results, indent=4))

Loading ONNX model...

Running MedMCQA...


100%|██████████| 500/500 [38:08<00:00,  4.58s/it]


medmcqa_accuracy =  0.374

Running PubMedQA...


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

1 / 1 | tokens= 399
1 / 2 | tokens= 386
2 / 3 | tokens= 261
2 / 4 | tokens= 373
3 / 5 | tokens= 302
4 / 6 | tokens= 352
4 / 7 | tokens= 305
4 / 8 | tokens= 325
4 / 9 | tokens= 244
5 / 10 | tokens= 532
6 / 11 | tokens= 295
6 / 12 | tokens= 399
6 / 13 | tokens= 393
7 / 14 | tokens= 343
8 / 15 | tokens= 307
9 / 16 | tokens= 493
10 / 17 | tokens= 379
11 / 18 | tokens= 173
12 / 19 | tokens= 319
13 / 20 | tokens= 380
14 / 21 | tokens= 362
15 / 22 | tokens= 238
16 / 23 | tokens= 340
17 / 24 | tokens= 376
17 / 25 | tokens= 334
18 / 26 | tokens= 236
19 / 27 | tokens= 378
19 / 28 | tokens= 368
20 / 29 | tokens= 224
21 / 30 | tokens= 236
21 / 31 | tokens= 314
21 / 32 | tokens= 281
22 / 33 | tokens= 148
23 / 34 | tokens= 392
24 / 35 | tokens= 328
25 / 36 | tokens= 323
26 / 37 | tokens= 335
26 / 38 | tokens= 424
27 / 39 | tokens= 281
28 / 40 | tokens= 480
29 / 41 | tokens= 453
30 / 42 | tokens= 395
30 / 43 | tokens= 222
30 / 44 | tokens= 408
30 / 45 | tokens= 269
31 / 46 | tokens= 342
32 / 47 | tok

KeyboardInterrupt: 

In [ ]:
import time
import json
import numpy as np

import onnxruntime_genai as og

from datasets import load_dataset
from tqdm import tqdm

# =========================
# CONFIG
# =========================

MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/apollo2-onnx/onnx-out/'

EVAL_SAMPLES = 500
MAX_LENGTH = 512

# =========================
# LOAD MODEL
# =========================

print("Loading ONNX model...")

model = og.Model(MODEL_PATH)

tokenizer = og.Tokenizer(model)

# =========================
# GENERATION FUNCTION
# =========================

def generate_answer(prompt):

    tokens = tokenizer.encode(prompt)
    input_length = len(tokens)

    # Create fresh params per request
    params = og.GeneratorParams(model)

    params.set_search_options(
        max_length=input_length + MAX_LENGTH,
        temperature=0.0,
        do_sample=False
    )

    generator = og.Generator(model, params)


    start = time.time()
    generator.append_tokens(tokens)

    while not generator.is_done():
            generator.generate_next_token()

    end = time.time()

    output_tokens = generator.get_sequence(0)

    text = tokenizer.decode(output_tokens)

    answer = text[len(prompt):].strip()

    latency = end - start

    #token_count = len(output_tokens)

    return answer, latency, input_length

def _process_logits_chunk(
    logits_list: list[np.ndarray],
    targets_list: list[int],
) -> tuple[float, int]:
    """Calculates NLL sum and count for a chunk of logits and targets."""
    chunk_logits = np.stack(logits_list)
    chunk_targets = np.array(targets_list)

    c = np.max(chunk_logits, axis=1, keepdims=True)
    lse = c + np.log(np.sum(np.exp(chunk_logits - c), axis=1, keepdims=True))

    row_indices = np.arange(len(chunk_targets))
    target_logits = chunk_logits[row_indices, chunk_targets]

    # Sum of NLL for this chunk
    # NLL = lse - target_logits
    chunk_nll_sum = np.sum(lse.squeeze() - target_logits)

    return float(chunk_nll_sum), len(chunk_targets)


def calculate_perplexity_onnxruntime_genai(text: str) -> float:
    """Calculates perplexity using optimized NumPy operations."""
    input_ids = tokenizer.encode(text)
    generator = og.Generator(model, og.GeneratorParams(model))

    input_ids_int32 = (
        input_ids.astype(np.int32) if input_ids.dtype != np.int32 else input_ids
    )

    logits_list = []
    targets_list = []
    append_tokens = generator.append_tokens
    get_logits = generator.get_logits

    nll_sum = 0.0
    count = 0
    chunk_size = 128

    # Iterate over input tokens to predict the next token
    # i goes from 0 to len-2.
    # input: input_ids_int32[i]
    # target: input_ids_int32[i+1]
    for i in range(len(input_ids_int32) - 1):
        append_tokens(input_ids_int32[i : i + 1])
        logits_list.append(get_logits()[0, 0, :])
        targets_list.append(int(input_ids_int32[i + 1]))

        if len(logits_list) >= chunk_size:
            # Process chunk
            chunk_nll_sum, chunk_count = _process_logits_chunk(
                logits_list,
                targets_list,
            )
            nll_sum += chunk_nll_sum
            count += chunk_count

            logits_list = []
            targets_list = []

    # Process remaining logits
    if logits_list:
        chunk_nll_sum, chunk_count = _process_logits_chunk(logits_list, targets_list)
        nll_sum += chunk_nll_sum
        count += chunk_count

    if count == 0:
        return 0.0

    return float(np.exp(nll_sum / count))

# =========================
# PUBMEDQA BENCHMARK
# =========================

print("\nRunning PubMedQA...")

pubmedqa = load_dataset(
    "openlifescienceai/pubmedqa",
    split="test"
)
results=np.zeros((500,3))
correct = 0
total = 301
for sample in pubmedqa.select(range(301,EVAL_SAMPLES)):

    context = " ".join(
        sample["data"]["Context"]
    )

    question = sample["data"]["Question"]

    prompt = f"""
Context:
{context}

Question:
{question}

Answer yes, no, or maybe.

Answer:
"""

    answer_text, latency, tokens = generate_answer(prompt)

    prediction = answer_text.lower()

    gold = sample["data"]["Correct Answer"].lower()

    if gold in prediction:
        correct += 1
        results[total,0]=1
    ppl = calculate_perplexity_onnxruntime_genai(prompt)

    total += 1
    print(total,",",correct,",",tokens,",",ppl)

Loading ONNX model...

Running PubMedQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


1 , 0 , 265 , 16.086209269907044
2 , 1 , 295 , 14.063927410003435
3 , 2 , 279 , 9.637827790394454
4 , 2 , 313 , 9.445624310804426
5 , 2 , 308 , 14.97743311017941
6 , 2 , 304 , 8.265302561496881
7 , 3 , 683 , 5.30062235415075
8 , 3 , 185 , 8.822400744637802
9 , 4 , 409 , 7.202155476229885
10 , 5 , 266 , 14.477486120149372
11 , 6 , 348 , 10.3402691070431
12 , 7 , 236 , 13.818625266379385
13 , 7 , 228 , 21.534007943539315
14 , 7 , 330 , 9.657382891100696
15 , 8 , 190 , 16.36061605616592
16 , 8 , 373 , 10.380929431711364
17 , 8 , 320 , 16.454610346604746
18 , 9 , 267 , 9.33805463032296
19 , 10 , 417 , 11.845192461784373
20 , 11 , 235 , 13.109810813509906
21 , 11 , 444 , 9.699134823262913
22 , 12 , 461 , 13.459582082575542
23 , 12 , 386 , 10.630531965724801
24 , 12 , 357 , 8.297541748592744
25 , 13 , 326 , 8.207722442060433
26 , 13 , 203 , 10.14185900114778
27 , 14 , 260 , 7.741450025501596
28 , 14 , 386 , 6.484257264465604
29 , 14 , 233 , 25.61487584436479
30 , 15 , 445 , 6.990915161392616

In [ ]:
import time
import json
import numpy as np

import onnxruntime_genai as og

from datasets import load_dataset
from tqdm import tqdm

# =========================
# CONFIG
# =========================

MODEL_PATH = './onnx-out/'

EVAL_SAMPLES = 500
MAX_LENGTH = 512

OUTPUT_FILE = "onnx_benchmark_results.json"

# =========================
# LOAD MODEL
# =========================

print("Loading ONNX model...")

model = og.Model(MODEL_PATH)

tokenizer = og.Tokenizer(model)

# =========================
# PROMPT FORMAT
# =========================

def format_medmcqa(sample):

    prompt = f"""
Question:
{sample['question']}

Options:
A. {sample['options']['A']}
B. {sample['options']['B']}
C. {sample['options']['C']}
D. {sample['options']['D']}

Answer:
"""

    return prompt


def extract_letter(text):

    for letter in ["A", "B", "C", "D"]:
        if letter in text:
            return letter

    return "A"

# =========================
# GENERATION FUNCTION
# =========================

def generate_answer(prompt):

    tokens = tokenizer.encode(prompt)
    input_length = len(tokens)

    # Create fresh params per request
    params = og.GeneratorParams(model)

    params.set_search_options(
        max_length=input_length + MAX_LENGTH,
        temperature=0.0,
        do_sample=False
    )

    generator = og.Generator(model, params)


    start = time.time()
    generator.append_tokens(tokens)

    while not generator.is_done():
            generator.generate_next_token()

    end = time.time()

    output_tokens = generator.get_sequence(0)

    text = tokenizer.decode(output_tokens)

    answer = text[len(prompt):].strip()

    latency = end - start

    #token_count = len(output_tokens)

    return answer, latency, input_length


# =========================
# MEDMCQA BENCHMARK
# =========================

print("\nRunning MedQA...")
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")
test_dataset = dataset["test"]
print("Test samples:", len(test_dataset))


correct = 0
tokens_total = 0

for sample in tqdm(test_dataset):

    prompt = format_medmcqa(sample)

    answer_text, latency, tokens = generate_answer(prompt)

    pred_letter = extract_letter(answer_text)

    correct_letter = sample["answer_idx"]

    if pred_letter == correct_letter:
        correct += 1


medmcqa_accuracy = correct / EVAL_SAMPLES

print("medqa_accuracy = ", medmcqa_accuracy )

Loading ONNX model...

Running MedQA...
Test samples: 1273


  0%|          | 6/1273 [01:08<3:46:21, 10.72s/it]